In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

symbol = "SOLUSDT"
interval = "1d"  # Денні свічки
limit = 500  # Останні 500 днів

url = f"https://data-api.binance.vision/api/v3/klines?symbol={symbol}&interval={interval}&limit={limit}"
response = requests.get(url)
data = response.json()

df = pd.DataFrame(data, columns=['timestamp', 'Open', 'High', 'Low', 'Close', 'Volume',
                                 'close_time', 'qav', 'num_trades', 'tbb', 'tbq',
                                 'ignore'])  #fields were generated by ai using api documentation
df = df[['timestamp', 'Open', 'High', 'Low', 'Close', 'Volume']].astype(float)
df['Date'] = pd.to_datetime(df['timestamp'], unit='ms')
df.set_index('Date', inplace=True)
print(df)

#Moving Average Crossover
short_ma = 20
long_ma = 50

df['Short_MA'] = df['Close'].rolling(window=short_ma).mean()
df['Long_MA'] = df['Close'].rolling(window=long_ma).mean()

df['Signal'] = 0
df['Signal'] = np.where(df['Short_MA'] > df['Long_MA'], 1,
                        0)  #1 is to buy, 0 to stay still. Where method can be used like IF
df['Position'] = df[
    'Signal'].diff()  # after last activity calculate difference to understand weather we need to buy, sell or do nothing (1, -1, 0)

               timestamp    Open    High     Low   Close       Volume
Date                                                                 
2025-03-11  1.741651e+12  118.32  128.43  112.00  125.35  6435277.863
2025-03-12  1.741738e+12  125.35  131.33  121.22  126.62  4410502.140
2025-03-13  1.741824e+12  126.61  128.76  120.76  123.37  2824584.961
2025-03-14  1.741910e+12  123.36  136.03  122.98  133.54  3607725.492
2025-03-15  1.741997e+12  133.53  136.53  132.44  135.86  1918077.490
...                  ...     ...     ...     ...     ...          ...
2026-07-19  1.784419e+12   75.53   76.70   75.37   76.38  1176910.446
2026-07-20  1.784506e+12   76.38   78.38   75.50   77.85  1867200.851
2026-07-21  1.784592e+12   77.85   78.88   77.42   78.12  1283561.320
2026-07-22  1.784678e+12   78.13   78.85   77.00   77.97  1424652.173
2026-07-23  1.784765e+12   77.97   78.54   77.40   77.69   168151.990

[500 rows x 6 columns]


In [ ]:
market_returns = [0.0]
historic_based = [0.0]
cumulated = [1.0] # initial capital

# convert df to list to make it easier
closes = list(df['Close'])
signals = list(df['Signal'])